In [1]:
pip install haversine

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U nbformat

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import json
from haversine import haversine, Unit
import plotly.express as px
import plotly.graph_objects as go

In [5]:
# load dataset
df_porto = pd.read_csv("porto/porto.csv")

In [6]:
# convert POLYLINE to list to make later analysis less complicated
df_porto["POLYLINE"] = df_porto["POLYLINE"].map(json.loads)

In [7]:
# avoid changing the original df
df = df_porto.copy(deep = True)
display(df) 


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643, 41.141412], [-8.618499, 41.141376..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847, 41.159826], [-8.640351, 41.159871..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964, 41.140359], [-8.613378, 41.14035]..."
3,1372636854620000520,C,NaN,NaN,20000520,1372636854,A,False,"[[-8.574678, 41.151951], [-8.574705, 41.151942..."
4,1372637091620000337,C,NaN,NaN,20000337,1372637091,A,False,"[[-8.645994, 41.18049], [-8.645949, 41.180517]..."
...,...,...,...,...,...,...,...,...,...
1710665,1404171463620000698,C,NaN,NaN,20000698,1404171463,A,False,"[[-8.612469, 41.14602], [-8.612487, 41.145993]..."
1710666,1404171367620000670,C,NaN,NaN,20000670,1404171367,A,False,"[[-8.610138, 41.140845], [-8.610174, 41.140935..."
1710667,1388745716620000264,C,NaN,NaN,20000264,1388745716,A,False,[]
1710668,1404141826620000248,B,NaN,12.0,20000248,1404141826,A,False,"[[-8.630712, 41.154885], [-8.63073, 41.154813]..."


## Part 1 - EDA
### Examine main attributes, get an overview of the data distribution and investigate the presence and impact of missing trajectories

#### TRIP_ID inspection

In [8]:
# check whether TRIP_ID is unique for each row
print("Number of unique TRIP_IDs:", len(df["TRIP_ID"].unique()))
print("Number of rows in the dataset:", len(df["TRIP_ID"]))
print("Number of duplicated TRIP_IDs:",  len(df["TRIP_ID"]) - len(df["TRIP_ID"].unique()))

# check how many rows are duplicated for each TRIP_ID and inspect trips with the most rows (value_counts sorts by TRIP_ID)
print("")
print("Number of duplicated rows for each TRIP_ID and inspection of the trip with the highest count:")
duplicate_ids_df = df[df["TRIP_ID"].duplicated(keep=False)].sort_values(by=["TRIP_ID"])
display(duplicate_ids_df.value_counts("TRIP_ID"))
display(duplicate_ids_df[duplicate_ids_df["TRIP_ID"] == 1397172149620000454 ])

# inspect the first 10 rows of the duplicated trips
print("First 10 rows of duplicated trips:")
display(duplicate_ids_df.head(10))

# find number of invalid rows in accrodance with part 2 task 7, less than three GPS points
display(f"The dtype of polyline is {type(duplicate_ids_df["POLYLINE"].iloc[0])}") 
invalid_duplicate_ids_df = duplicate_ids_df[duplicate_ids_df["POLYLINE"].str.len() < 3]
display(invalid_duplicate_ids_df)

# check how many duplicates there are when removing invalid trips
valid_ids_df = df.drop(invalid_duplicate_ids_df.index)
print("Number of unique TRIP_IDs:", len(df["TRIP_ID"].unique()))
print("Number of rows in the new dataset:", len(valid_ids_df["TRIP_ID"]))
print("Number of duplicated TRIP_IDs:",  len(valid_ids_df["TRIP_ID"]) - len(df["TRIP_ID"].unique()))

Number of unique TRIP_IDs: 1710589
Number of rows in the dataset: 1710670
Number of duplicated TRIP_IDs: 81

Number of duplicated rows for each TRIP_ID and inspection of the trip with the highest count:


TRIP_ID
1397172149620000454    3
1372702836620000080    2
1373210896620000598    2
1373025987620000601    2
1374014097620000337    2
                      ..
1403115272620000066    2
1403323491620000600    2
1403407180620000242    2
1403716904620000337    2
1404093316620000307    2
Name: count, Length: 80, dtype: int64

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
1304113,1397172149620000454,A,30929.0,NaN,20000454,1397172149,A,False,"[[-8.618391, 41.162067], [-8.6184, 41.161446],..."
1304114,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391, 41.162067], [-8.6184, 41.161446],..."
1304115,1397172149620000454,C,NaN,NaN,20000454,1397172149,A,False,"[[-8.618391, 41.162067], [-8.6184, 41.161446],..."


First 10 rows of duplicated trips:


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
3519,1372702836620000080,B,NaN,16.0,20000080,1372702836,A,False,"[[-8.635275, 41.193729], [-8.634519, 41.192901]]"
3720,1372702836620000080,C,NaN,NaN,20000080,1372702836,A,False,"[[-8.62776, 41.17554], [-8.62776, 41.17554], [..."
21282,1373025987620000601,A,2002.0,NaN,20000601,1373025987,A,False,[]
21385,1373025987620000601,C,NaN,NaN,20000601,1373025987,A,False,"[[-8.604108, 41.182569], [-8.604072, 41.182605..."
33228,1373210896620000598,B,NaN,57.0,20000598,1373210896,A,False,"[[-8.610885, 41.145525]]"
33350,1373210896620000598,C,NaN,NaN,20000598,1373210896,A,False,"[[-8.610813, 41.14548], [-8.610732, 41.145579]..."
51623,1373551710620000014,B,NaN,49.0,20000014,1373551710,A,False,"[[-8.587278, 41.147325], [-8.587278, 41.147325]]"
51966,1373551710620000014,C,NaN,NaN,20000014,1373551710,A,False,"[[-8.602938, 41.179716], [-8.603937, 41.179932..."
76527,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648451, 41.164785]]"
76596,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648478, 41.164758], [-8.648478, 41.164758..."


"The dtype of polyline is <class 'list'>"

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
3519,1372702836620000080,B,NaN,16.0,20000080,1372702836,A,False,"[[-8.635275, 41.193729], [-8.634519, 41.192901]]"
21282,1373025987620000601,A,2002.0,NaN,20000601,1373025987,A,False,[]
33228,1373210896620000598,B,NaN,57.0,20000598,1373210896,A,False,"[[-8.610885, 41.145525]]"
51623,1373551710620000014,B,NaN,49.0,20000014,1373551710,A,False,"[[-8.587278, 41.147325], [-8.587278, 41.147325]]"
76527,1374014097620000337,C,NaN,NaN,20000337,1374014097,A,False,"[[-8.648451, 41.164785]]"
...,...,...,...,...,...,...,...,...,...
1647671,1403115272620000066,C,NaN,NaN,20000066,1403115272,A,False,"[[-8.554752, 41.172975]]"
1665292,1403407180620000242,C,NaN,NaN,20000242,1403407180,A,False,"[[-8.610687, 41.14773], [-8.611722, 41.147856]]"
1665302,1403407180620000242,C,NaN,NaN,20000242,1403407180,A,False,[]
1705646,1404093316620000307,B,NaN,57.0,20000307,1404093316,A,False,"[[-8.610831, 41.145741]]"


Number of unique TRIP_IDs: 1710589
Number of rows in the new dataset: 1710564
Number of duplicated TRIP_IDs: -25


In [9]:
# Get statistics on POLYLINE length for first occurance and last occurance:
# For the TRIP_ID with three rows, we saw that the two last are exactly the same. We cantherefore remove duplicates
duplicate_ids_df_filtered = duplicate_ids_df.astype(str).drop_duplicates()
display(duplicate_ids_df.shape, duplicate_ids_df_filtered.shape)

# undo convertion to str and calculate length
duplicate_ids_df_filtered["POLYLINE_LENGTH"] = duplicate_ids_df_filtered["POLYLINE"].str.len()

# Find occurences where the last have fewer GPS points than the first
duplicate_ids_df_filtered_deviation = duplicate_ids_df_filtered.groupby("TRIP_ID")["POLYLINE_LENGTH"].agg(lambda x: (x.iloc[-1] - x.iloc[0])).reset_index()
display(duplicate_ids_df_filtered_deviation)
ids_with_negative_deviation = duplicate_ids_df_filtered_deviation[duplicate_ids_df_filtered_deviation["POLYLINE_LENGTH"] < 0]["TRIP_ID"]
display(duplicate_ids_df_filtered[duplicate_ids_df_filtered["TRIP_ID"].isin(ids_with_negative_deviation) ])

(161, 9)

(158, 9)

,TRIP_ID,POLYLINE_LENGTH
0,1372702836620000080,1240
1,1373025987620000601,877
2,1373210896620000598,429
3,1373551710620000014,1542
4,1374014097620000337,1759
...,...,...
75,1403115272620000066,48
76,1403323491620000600,309
77,1403407180620000242,-45
78,1403716904620000337,2520


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE,POLYLINE_LENGTH
494048,1381859675620000465,C,nan,nan,20000465,1381859675,A,False,"[[-8.631909, 41.171958]]",24
494052,1381859675620000465,C,nan,nan,20000465,1381859675,A,False,"[[-8.628561, 41.17266]]",23
707187,1385772222620000263,B,nan,15.0,20000263,1385772222,A,False,"[[-8.585685, 41.148522], [-8.585694, 41.148531]]",48
707193,1385772222620000263,C,nan,nan,20000263,1385772222,A,False,[],2
769848,1386942349620000080,C,nan,nan,20000080,1386942349,A,False,"[[-8.587188, 41.167017]]",24
769850,1386942349620000080,C,nan,nan,20000080,1386942349,A,False,"[[-8.58492, 41.166504]]",23
844671,1388279758620000080,C,nan,nan,20000080,1388279758,A,False,"[[-8.589627, 41.163732]]",24
844672,1388279758620000080,C,nan,nan,20000080,1388279758,A,False,"[[-8.58537, 41.162463]]",23
1063516,1392640481620000663,C,nan,nan,20000663,1392640481,A,False,"[[-8.604684, 41.158107]]",24
1063517,1392640481620000663,C,nan,nan,20000663,1392640481,A,False,"[[-8.60481, 41.158989]]",23


### Conclusion:
It appears that 81 trips have does not have unique TRIP_IDs and that the rows with similar TRIP_ID are not simply duplicates of one another. It appears that for most cases there are two duplicate rows, but there is one case where there are three duplicate rows. For the duplicated rows TAXI_ID and timestamp appear to be the same. This could indicate that the first initiaton of saving the trip failed and therefore the the second or third is an attempt to correct the error. For example some of the first TRIP_IDs initiated have were few coordinates in the POLYLINE field even though MISSING_DATA is false. We cannot keep both ids as this will cause issues when creating SQL tables with TRIP_ID as primary key. After inspecting the data, it appears that for most cases the first TRIP_ID occurences are missing data points except for some cases where the alst are missing data points. We have therefore decided to only keep the row with the most POLYLINE points, for rows with duplicate TRIP_ID. 

It also became clear that POLYLINE was read as a str. When converting it to a list, we were able to check how many of the ids that appeared multiple times in the dataset had fewer than three GPS points. After removing the ids that appeared multiple times with less than thre GPS points, we ended up with 25 fewer ids.



#### CALL_TYPE, ORIGIN_CALL and ORIGIN_STAND inspection

In [10]:
# check that there are in fact only A, B and C values for calltype
display(df["CALL_TYPE"].unique())

# function to check inconsistency in CALL_TYPE, ORIGIN_CALL and ORIGIN_STAND
def calltype_validation(column_name, call_type):
    invalid_rows = df[
                            ((df[column_name].isna()) & (df["CALL_TYPE"]==call_type)) | 
                            ((df[column_name].notna()) & (df["CALL_TYPE"]!=call_type))
                    ]
    invalid_rows_na = df[((df[column_name].isna()) & (df["CALL_TYPE"]==call_type))]
    invalid_rows_not_na = df[((df[column_name].notna()) & (df["CALL_TYPE"]!=call_type))]
    display(invalid_rows) 
    if invalid_rows.shape[0] != 0:
        print(f"Number of rows where CALL_TYPE={call_type} and {column_name} is NaN: {invalid_rows_na.shape[0]}") 
        print(f"Number of rows where CALL_TYPE!={call_type} and {column_name} is not NaN: {invalid_rows_not_na.shape[0]}") 
        return
    print(f"No inconsistency between CALL_TYPE and {column_name}")
    
# ORIGIN_CALL
calltype_validation("ORIGIN_CALL", "A")

# ORIGIN_STAND
calltype_validation("ORIGIN_STAND", "B")
# check that 

array(['C', 'B', 'A'], dtype=object)

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE


No inconsistency between CALL_TYPE and ORIGIN_CALL


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
136,1372638560620000308,B,NaN,NaN,20000308,1372638560,A,False,"[[-8.58564, 41.148567], [-8.585667, 41.148873]..."
280,1372650696620000320,B,NaN,NaN,20000320,1372650696,A,False,"[[-8.6058, 41.153517], [-8.606133, 41.153472],..."
433,1372656129620000307,B,NaN,NaN,20000307,1372656129,A,False,"[[-8.63064, 41.154714], [-8.630685, 41.154102]..."
564,1372665032620000004,B,NaN,NaN,20000004,1372665032,A,False,"[[-8.591319, 41.156181], [-8.591463, 41.156217..."
640,1372663170620000391,B,NaN,NaN,20000391,1372663170,A,False,"[[-8.630712, 41.154966], [-8.630775, 41.154867..."
...,...,...,...,...,...,...,...,...,...
1710248,1404167874620000600,B,NaN,NaN,20000600,1404167874,A,False,"[[-8.617545, 41.14611], [-8.617563, 41.146083]..."
1710472,1404151531620000001,B,NaN,NaN,20000001,1404151531,A,False,"[[-8.605638, 41.153589], [-8.607051, 41.153589..."
1710480,1404172241620000675,B,NaN,NaN,20000675,1404172241,A,False,"[[-8.617662, 41.146083], [-8.617626, 41.145984..."
1710639,1404155788620000591,B,NaN,NaN,20000591,1404155788,A,False,"[[-8.585586, 41.14863], [-8.585694, 41.148765]..."


Number of rows where CALL_TYPE=B and ORIGIN_STAND is NaN: 11302
Number of rows where CALL_TYPE!=B and ORIGIN_STAND is not NaN: 0


In [11]:
# interpolate point B based on trips that have been done within a close distanse
df_ORIGIN_STAND_B = df[df["CALL_TYPE"]=="B"].copy()

def polyline_to_lat_and_lon(df):
    df["LONGITUDE"] = df["POLYLINE"].apply( lambda x: x[0][0] if len(x) > 0 else None)
    df["LATITUDE"] = df["POLYLINE"].apply( lambda x: x[0][1] if len(x) > 0 else None)
    return df

# Trips with missing value for ORIGIN_STAND
missing_origin_stand = df_ORIGIN_STAND_B[df_ORIGIN_STAND_B["ORIGIN_STAND"].isna()][["TRIP_ID", "ORIGIN_STAND", "POLYLINE"]].copy()
missing_origin_stand = polyline_to_lat_and_lon(missing_origin_stand)


# Keep one Trip for each stand as reference points for the rest of the trips without ORIGIN_STAND
origin_stand_trips = df_ORIGIN_STAND_B[~df_ORIGIN_STAND_B["ORIGIN_STAND"].isna()].copy()
origin_stand_trips = polyline_to_lat_and_lon(origin_stand_trips)
origin_stand_trips_median = origin_stand_trips.groupby("ORIGIN_STAND").agg(LATITUDE=("LATITUDE", "median"),LONGITUDE=("LONGITUDE", "median")).reset_index() # use median value to avoid outliers

In [12]:
# plot all stands and all trips with stands for inspection
fig = px.scatter_map(
    origin_stand_trips_median,
    lat="LATITUDE",
    lon="LONGITUDE",
    custom_data="ORIGIN_STAND",
    zoom=12
)

fig.update_traces(
    name="Contains ORIGIN_STAND",
    marker=dict(size=18),
    hovertemplate=(
        "ORIGIN_STAND: %{customdata}<br>"
        "Lat: %{lat}<br>"
        "Lon: %{lon}"
        "<extra></extra>"
    ),
    selector=0
)

fig.add_trace(
    go.Scattermap(
        lat=missing_origin_stand["LATITUDE"],
        lon=missing_origin_stand["LONGITUDE"],
        mode="markers",
        name="Missing ORIGIN_STAND",
        marker=dict(size=8),
        customdata=missing_origin_stand["TRIP_ID"],
        hovertemplate=(
            "TRIP_ID: %{customdata}<br>"
            "Lat: %{lat}<br>"
            "Lon: %{lon}"
            "<extra></extra>"
        )
    )
)

# Put legend on the right
fig.update_layout(
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show()

### Conclusion

CALL_TYPE only contains the values A, B, C as expected. 

There are no inconsitencies between CALL_TYPE and ORIGIN_CALL as all ORIGIN_CALL is only NaN when CALLTYPE is not A and it is never NaN when CALL_TYPE is A.
There are 11302 rows where CALL_TYPE is B and ORIGIN_STAND is not NaN and 0 rows where where CALL_TYPE!=B and ORIGIN_STAND is not NaN. This is not necessarily a problem and could simply be due to lacking information about start. This could be interpolated by analysing the start from the POLYLINE column.

With further analysis, it appears that not all trips are correctly catagorized as CALL_TYPE=B since some red dots are not close to an ORIGIN_STAND (blue dots) in the plot. Since we do not know if they should have been categorized as A or C, and as the later questions in Part 2 does not depend on correct CALL_TYPE we have decided to leave it as is. 

#### Unique TAXI_ID inspection - Check that the same taxi cannot do two trips at once

In [13]:
# Calculate the end of each trip based on POLYLINE
df["LENGTH"] = 15*(df["POLYLINE"].str.len() - 1).clip(lower=0) # clip to avoid -15 seconds and subtract 1 since the first geopoint is sampled after 0 seconds
df["TIMESTAMP_END"] = df["TIMESTAMP"] + df["LENGTH"]

# sort values by TAXI_ID and TIMESTAMP and check for overlapping trips done by the same taxi
porto_df_by_taxi = df.sort_values(by=["TAXI_ID", "TIMESTAMP"])
overlap = (porto_df_by_taxi["TAXI_ID"] == porto_df_by_taxi["TAXI_ID"].shift(1)) & (porto_df_by_taxi["TIMESTAMP_END"].shift(1) > porto_df_by_taxi["TIMESTAMP"])
porto_df_by_taxi["OVERLAP"] = overlap | overlap.shift(-1, fill_value=False)

# for inspection of result
display(porto_df_by_taxi[porto_df_by_taxi["OVERLAP"]].sort_values(by = ["TAXI_ID", "TIMESTAMP"]))

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE,LENGTH,TIMESTAMP_END,OVERLAP
1110508,1393503462620000002,C,NaN,NaN,20000002,1393503462,A,False,"[[-8.611704, 41.16537], [-8.611236, 41.16474],...",45,1393503507,True
1110905,1393503496620000002,C,NaN,NaN,20000002,1393503496,A,False,"[[-8.611704, 41.16537], [-8.611236, 41.16474],...",1305,1393504801,True
339317,1379242313620000004,A,2002.0,NaN,20000004,1379242313,A,False,"[[-8.57475, 41.142753], [-8.574759, 41.142771]...",1380,1379243693,True
339407,1379243647620000004,C,NaN,NaN,20000004,1379243647,A,False,"[[-8.615214, 41.141916], [-8.615232, 41.141772...",1530,1379245177,True
1301641,1397131137620000004,B,NaN,2.0,20000004,1397131137,A,False,"[[-8.591292, 41.156208], [-8.591481, 41.156217...",735,1397131872,True
...,...,...,...,...,...,...,...,...,...,...,...,...
407522,1380378110620000904,C,NaN,NaN,20000904,1380378110,A,False,"[[-8.586351, 41.147748], [-8.58618, 41.148045]...",3660,1380381770,True
1265287,1396453938620000904,C,NaN,NaN,20000904,1396453938,A,False,"[[-8.607456, 41.14548], [-8.607447, 41.145507]...",285,1396454223,True
1265681,1396454206620000904,C,NaN,NaN,20000904,1396454206,A,False,"[[-8.607177, 41.145417], [-8.607213, 41.145426...",1725,1396455931,True
1490010,1400411434620000904,C,NaN,NaN,20000904,1400411434,A,False,"[[-8.633403, 41.171679], [-8.629011, 41.172588...",1920,1400413354,True


### Conclusion
There appear to be 1714 rows in which the same TAXI_ID has some overlap between two different trips. There appears to only be some seconds difference. We therefore assumed that this is due to clocking errors and not actual anomalies in the data.

### TIMESTAMP inspection

In [14]:
# Start time of the trip, expressed as Unix time (seconds since 1970-01-01). Should be converted to a standard DATETIME format for queries.
df["DATE_AND_HOUR"] = pd.to_datetime(df["TIMESTAMP"], unit="s")
min_time = df["DATE_AND_HOUR"].min()
max_time = df["DATE_AND_HOUR"].max()
display(df["DATE_AND_HOUR"])
print(f"The first timestamp is at {min_time} whereas the last timestamp is at {max_time}")

0         2013-07-01 00:00:58
1         2013-07-01 00:08:23
2         2013-07-01 00:02:31
3         2013-07-01 00:00:54
4         2013-07-01 00:04:51
                  ...        
1710665   2014-06-30 23:37:43
1710666   2014-06-30 23:36:07
1710667   2014-01-03 10:41:56
1710668   2014-06-30 15:23:46
1710669   2014-06-30 19:39:07
Name: DATE_AND_HOUR, Length: 1710670, dtype: datetime64[ns]

The first timestamp is at 2013-07-01 00:00:53 whereas the last timestamp is at 2014-06-30 23:59:56


### DAYTIME inspection

In [15]:
# Indicates whether the trip occurred on a normal day, a holiday, or the day before a holiday. A: Normal day (workday or weekend), B: Holiday or special day, C: Day before a holiday.
display(df["DAY_TYPE"].unique())

array(['A'], dtype=object)

### MISSING_DATA inspection

In [16]:
# Boolean flag indicating whether there are missing GPS points in the POLYLINE. FALSE means the data is complete, TRUE means some points are missing.
display(df["MISSING_DATA"].unique())
missing_data = df[df["MISSING_DATA"]]

array([False,  True])

In [17]:
df_explode_popyline = missing_data[["TRIP_ID", "POLYLINE"]].explode("POLYLINE").copy()

df_explode_popyline[["Longitude", "Latitude"]] = pd.DataFrame(
    df_explode_popyline["POLYLINE"].tolist(),
    index=df_explode_popyline.index
)

df_explode_popyline = df_explode_popyline.drop(columns="POLYLINE").reset_index(drop=True)

display(df_explode_popyline)

,TRIP_ID,Longitude,Latitude
0,1374554455620000625,-8.612559,41.145975
1,1374554455620000625,-8.612577,41.145975
2,1374554455620000625,-8.612586,41.145966
3,1374554455620000625,-8.612559,41.145984
4,1374554455620000625,-8.612127,41.145966
...,...,...,...
964,1399405185620000508,-8.572302,41.141268
965,1399405185620000508,-8.572446,41.143158
966,1399405185620000508,-8.573985,41.144004
967,1399405185620000508,-8.576568,41.144193


In [18]:
# Remove trips with Polyline lenght less than 1. 
# unique trip_ids with missing polyline points
ids_missing_data = missing_data["TRIP_ID"].unique()

print(f"The TRIP_IDs containing missing data are: {ids_missing_data}")


fig = go.Figure()

for trip_id, trip in df_explode_popyline.groupby("TRIP_ID"):

    trip = trip.reset_index(drop=True) # reset index to keep track ovof GPS data points

    fig.add_trace(
        go.Scattermap(
            lat=trip["Latitude"],
            lon=trip["Longitude"],
            customdata=trip.index,
            mode="lines",
            name=str(trip_id),
            hovertemplate=(
                f"Trip: {trip_id}<br>"
                "GPS nr: %{customdata}<br>"
                "Lat: %{lat}<br>"
                "Lon: %{lon}"
                "<extra></extra>"
            )
        )
    )

fig.update_layout(
    map=dict(
        style="basic",
        zoom=12,
        center={
            "lat": df_explode_popyline["Latitude"].mean(),
            "lon": df_explode_popyline["Longitude"].mean()
        }
    ),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
    title="Trips"
)

fig.show()

The TRIP_IDs containing missing data are: [1374554455620000625 1375863510620000454 1378544246620000057
 1381233613620000387 1386346894620000904 1387137779620000640
 1388351478620000678 1390005983620000640 1396631707620000163
 1399405185620000508]


Through inspection we can find exactly which datapoints are missing and how servere the missing data is:

- 1374554455620000625: Does not seem to lack many data points, and the trip is rather short
- 1375863510620000454: There is only one GPS-point
- 1378544246620000057: Appear to miss quite a lot of data points 402 and 403
- 1381233613620000387: Long trip with many data points, but there are a lot of points missing between point 
- 1386346894620000904: Long trip with many data points, but there are a lot of points missing between points 261 to 280
- 1387137779620000640: Short trip containing most data points, but there are some points missing between point 39 and 40
- 1388351478620000678: Short trip missing some points here and there
- 1390005983620000640: Short trip missing some points here and there
- 1396631707620000163: Only contains one GPS point
- 1399405185620000508: Short trup missing some GPS points

We have decided not to remove any data, but many missing data points for certain trips means we have to be careful when analysing the time spent travelling based this shoul dbe done based on distance and not based on number of GPS point 

# Cleaning data

In [19]:
# keep trip with longest POLYLINE for duplicate TRIP_IDs
df_porto["POLYLINE_LENGTH"] = df_porto["POLYLINE"].str.len()
df_porto = df_porto.sort_values("POLYLINE_LENGTH", ascending=False).drop_duplicates(subset=['TRIP_ID'], keep='first')
df_porto = df_porto.drop(columns=["POLYLINE_LENGTH"])
display(df_porto)

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
1093727,1393061275620000681,C,NaN,NaN,20000681,1393061275,A,False,"[[-8.583435, 41.158665], [-8.582958, 41.158899..."
1492417,1400312076620000562,C,NaN,NaN,20000562,1400312076,A,False,"[[-8.614881, 41.151051], [-8.614836, 41.151087..."
578581,1383293324620000520,B,NaN,13.0,20000520,1383293324,A,False,"[[-8.628237, 41.157558], [-8.628012, 41.158035..."
224510,1376892235620000086,C,NaN,NaN,20000086,1376892235,A,False,"[[-8.618148, 41.167557], [-8.618409, 41.167575..."
147121,1375289927620000902,C,NaN,NaN,20000902,1375289927,A,False,"[[-8.614386, 41.141079], [-8.613729, 41.141304..."
...,...,...,...,...,...,...,...,...,...
12120,1372864273620000319,C,NaN,NaN,20000319,1372864273,A,False,[]
1532656,1401185258620000021,C,NaN,NaN,20000021,1401185258,A,False,[]
333098,1379157792620000496,C,NaN,NaN,20000496,1379157792,A,False,[]
12079,1372865745620000507,B,NaN,11.0,20000507,1372865745,A,False,[]


In [20]:
# convert to datetime



In [21]:
# write to csv - create one csv with all the data and one with only TRIP_ID and 
df_explode_popyline = missing_data[["TRIP_ID", "POLYLINE"]].explode("POLYLINE").copy()

df_explode_popyline[["Longitude", "Latitude"]] = pd.DataFrame(
    df_explode_popyline["POLYLINE"].tolist(),
    index=df_explode_popyline.index
)

df_explode_popyline = df_explode_popyline.drop(columns="POLYLINE").reset_index(drop=True)

display(df_explode_popyline)

,TRIP_ID,Longitude,Latitude
0,1374554455620000625,-8.612559,41.145975
1,1374554455620000625,-8.612577,41.145975
2,1374554455620000625,-8.612586,41.145966
3,1374554455620000625,-8.612559,41.145984
4,1374554455620000625,-8.612127,41.145966
...,...,...,...
964,1399405185620000508,-8.572302,41.141268
965,1399405185620000508,-8.572446,41.143158
966,1399405185620000508,-8.573985,41.144004
967,1399405185620000508,-8.576568,41.144193
